# FastGS — training and submission pipeline

This notebook is **glue only**. Every step is a function in `pipeline/`:

| File | Responsibility |
|---|---|
| `pipeline/config.py` | every parameter, in one place |
| `pipeline/env.py` | GPU check, dependency install, RAM/VRAM tracking and cleanup |
| `pipeline/data.py` | download data, discover scenes, data profile |
| `pipeline/trainer.py` | FastGS training loop with live scoring |
| `pipeline/score.py` | LPIPS / SSIM / PSNR and `Score = 0.4(1−LPIPS) + 0.3·SSIM + 0.3·PSNR_norm` |
| `pipeline/submission.py` | render the test poses → `submission.zip` + format check |
| `pipeline/report.py` | comparison tables and plots |
| `pipeline/deliver.py` | package results and download them |
| `pipeline/run.py` | wires it together (`setup → load_data → smoke_test → run_all → analytics → finish`) |

How FastGS works: [DOCS/fastgs-acceleration-method.md](DOCS/fastgs-acceleration-method.md) — with a runnable,
CUDA-free micro-simulation in `python demos/fastgs_mechanisms.py`.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

## 0 — Get the code

The first install takes ~3–5 minutes to build the three CUDA submodules; later sessions skip it via the `/content/.deps_ok` flag.

In [ ]:
REPO_URL = "https://github.com/KietAnhCS/fastgs-lite.git"
REPO_DIR = "/content/fastgs-lite"

import os, sys

if os.path.isdir("pipeline"):
    REPO_DIR = os.getcwd()                      # already running inside the repo
elif not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

## 1 — Check the GPU

In [ ]:
from pipeline.env import check_gpu, install_dependencies

install_dependencies()        # pip packages + 3 CUDA submodules (skipped once installed)
check_gpu(require=True)       # GPU / VRAM / torch / CUDA / RAM

## 2 — Configuration

Edit **here only**. For the competition data, either set `dataset_url` to the dataset ZIP or upload the
scenes into `data_root` yourself; leave `scenes=()` and the pipeline discovers every scene it finds.

In [ ]:
from pipeline import Config, run, report

cfg = Config(
    data_root="/content/data",
    # dataset_url=None,            # None = data is already sitting in data_root
    scenes=(),                     # () = use every scene found
    resolution=2,                  # training images downscaled 2x to fit a T4
    iterations=7000,
    score_every=1000,              # how often to score during training
    eval_views=6,
    psnr_max=30.0,                 # PSNR_max used by the organisers
    submission_resolution=1,       # render at the original image size
)
cfg.show()

## 3 — Load the dataset and profile it

In [ ]:
scenes, profile = run.load_data(cfg)
display(profile)

## 4 — Quick test (one scene, a few hundred iterations)

Proves data + CUDA + scoring all work before spending hours on the real run.

In [ ]:
run.smoke_test(cfg, scenes[0])

## 5 — Train

The progress bar shows **percent complete**, loss, Gaussian count and the latest **Score**. Every
`score_every` iterations a full line is printed: Score and its change, PSNR (with `psnr_norm`), SSIM,
LPIPS, RAM and VRAM. Each scene is rendered to the submission folder as soon as it finishes training,
then its memory is released for the next scene.

In [ ]:
results, submissions = run.run_all(cfg, scenes)

## 6 — Data analytics: compare the scenes

In [ ]:
history, board = run.analytics(cfg, results, submissions)
display(board)

In [ ]:
report.show_samples(cfg, scenes[0], n=3)      # renders next to ground truth

## 7 — Build `submission.zip` and download it

```
submission.zip
├── <scene>/0001.png, 0002.png, ...
└── ...
```

`run.finish` zips the renders, **validates the format** (scene names, contiguous file names, image count
per scene, image size), then downloads `submission.zip` — plus the packaged models and reports — to your
machine automatically.

In [ ]:
check, problems = run.finish(cfg, scenes)
display(check)